In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import pathlib
import dotenv
import pickle
import sys
import json5
import csv
import os
import pandas as pd
import numpy as np
from typing import Dict, List
from ast import literal_eval
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError, parse_obj_as

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger

# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.description_v2 import (
    extract_metadata_for_urn,
    transform_table_info_for_llm,
)
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)

from docs_generation.graph_helper import create_datahub_graph
from helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict,
    get_failed_response_table_urns,
    read_labeled_column_data,
    get_prediction_df,
    filter_predictions_df,
    get_classification_report_df,
)

# TERM_SUGGESTION_GENERATION_MODEL: BedrockModel = parse_obj_as(
#     BedrockModel,
#     os.getenv(
#         "TERM_SUGGESTION_GENERATION_BEDROCK_MODEL", BedrockModel.CLAUDE_3_HAIKU.value
#     ),
# )

dotenv.load_dotenv(current_dir / ".env")

True

In [ ]:
experiment_dict = [
    {
        "prompt_path": "prompts/Claude_10_no_example.txt",
        "experiment_name": "Claude_10_no_example",
        "iterations": 3,
        "confidence_thresholds": [5, 6, 7, 8, 8.5, 9, 9.5, 10],
    },
    {
        "prompt_path": "prompts/Claude_10.txt",
        "experiment_name": "Claude_10",
        "iterations": 3,
        "confidence_thresholds": [5, 6, 7, 8, 8.5, 9, 9.5, 10],
    },
]

In [5]:
BASE_DIRECTORY = current_dir / f"Care_new_{datetime.now().strftime('%m-%d-%H-%M')}/"
COLUMN_LABELS_CSV_PATH = current_dir / "care_columns_ground_truth_without_comment.csv"
URNS_DICT_PATH = current_dir / "test_urns.json"
INSTANCES_TO_EXAMINE = ["care"]
GLOSSARY_INSTANCE = "care"
GLOSSARY_NODES = [
    "urn:li:glossaryNode:d966ad51-49a7-411c-8d06-2ee2dd210647",
    "urn:li:glossaryNode:09effdec-8810-415c-ace7-4e53090c502c",
]
GLOSSARY_TERMS = []
LABEL_COLUMN = "glossary_terms_updated_new"
URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
    (URNS_DICT_PATH).read_text()
)
FILTERS = [
    "no_filter",
    "description",
    "sample_values",
    "description_and_sample_values",
    "description_or_sample_values",
    "name_and_datatype",
]
URNS_DICT = {
    key: value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE
}
# CONFIDENCE_THRESHOLDS = [1,2,3,4] #[7, 7.5, 8, 8.5, 9, 9.5] # [1,2,3,4,5] #
# URND_DICT = {"longtailcompanions": URNS_DICT["longtailcompanions"][0]}

In [31]:
tables_info_dict, columns_info_dict = get_table_and_column_infos_dict(
    urns_dict=URNS_DICT
)

urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_lake.dl_akamai.dl_akamai_logs,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_bmw.adi_chl.dw_f_case_intl,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_data_lake.dom_czen.sitter_activity,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_datascience.datascience.provider_responsiveness_predictions,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_data_lake.salesforce.user_app_info,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_data_lake.salesforce.user_team_member,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_bmw.test.account_history,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_bmw.history_vertica.dw_f_event,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_bmw.salesforce_sf.account,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_data_marts.rpt_reporting.dw_d_interstitial,PROD)
urn:li:dataset:(urn:li:da

In [8]:
# graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
# glossary_config = GlossaryUniverseConfig(
#     glossary_nodes=GLOSSARY_NODES,
#     glossary_terms=GLOSSARY_TERMS
# )
# glossary_info = fetch_glossary_info(graph_client=graph_client, universe=glossary_config)

# print(len(glossary_info.glossary.keys()))
# terms_to_remove = ['Day', 'Month', 'Year']
# glossary_info.glossary = {key: value for key, value in glossary_info.glossary.items() if value["term_name"] not in terms_to_remove}
# print(len(glossary_info.glossary.keys()))

with open(
    "care_glossary_info_with_examples_and_user_and_member_id_descriptions.pkl", "rb"
) as f:
    glossary_info = pickle.load(f)

In [9]:
terms = []
for key, value in glossary_info.glossary.items():
    terms.append(value["term_name"])
print(terms)

['Allergies', 'Password - Encrypted', 'IBAN', 'Credit Card Token', 'Password - Cleartext', 'Driving License Number', 'Government ID', 'Tax ID', 'Social Security Number', 'User ID', 'Health Information', 'Credit / Debit Card Number', 'Bank Routing Number', 'Bank Account Number', 'Authentication Token', 'Username', 'First Name', 'Middle Name', 'Full Name', 'Person Name', 'Last Name', 'IP Address', 'Member Messages', 'Geolocation', 'Social Media ID', 'Gender', 'Personal Photo', 'Employee ID', 'Geo Location - Coarse', 'Geo Location - Precise', 'Phone Number', 'Income', 'Email Address', 'County', 'City', 'Address', 'State', 'Zip Code', 'Street Address', 'Country', 'Marital Status', 'Member ID', 'Birth Day', 'Birth Year', 'Birth Date', 'Birth Month', 'Member UUID']


In [13]:
from datahub_integrations.gen_ai.description_v2 import (
    generate_entity_descriptions_for_urn,
    parse_llm_output,
)

In [11]:
graph_client = create_datahub_graph("care")
entity_descriptions = {}
for urn in URNS_DICT["care"]:
    entity_descriptions[urn] = generate_entity_descriptions_for_urn(
        graph_client=graph_client, urn=urn
    )

2024-09-25 11:59:55.302 | INFO     | datahub_integrations.gen_ai.bedrock:get_bedrock_client:36 - Initializing Bedrock client from explicit env vars
2024-09-25 12:00:04.140 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 8.837796449661255 seconds
2024-09-25 12:00:08.433 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 3.5890920162200928 seconds
2024-09-25 12:00:13.766 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 4.76413631439209 seconds
2024-09-25 12:00:18.872 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 4.526484727859497 seconds
2024-09-25 12:00:23.079 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 3.5586719512939453 seconds
2024-09-25 12:00:28.106 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 4.424821615219116 seconds
2024-09-25 12:00:33.158 | DEBUG    | data

In [17]:
parsed_entity_descriptions = {}
for key, value in entity_descriptions.items():
    #     print(value)
    #     break
    parsed_entity_descriptions[key] = parse_llm_output(value[0])

In [26]:
parsed_entity_descriptions

{'urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_lake.dl_akamai.dl_akamai_logs,PROD)': ('# dl_akamai_logs\n\nThis table contains logs from the Akamai content delivery network (CDN) used to serve content for the company\'s websites and applications. The data in this table is considered a "Large Table" based on its storage size footprint within the data platform over the past 30 days.\n\nThe data in this table is ingested directly from the Akamai CDN and does not undergo any transformations prior to landing in this table. This table serves as a raw, unprocessed data source for downstream analytics and reporting use cases.\n\nConsumers of this data may include the website and application teams, the infrastructure and operations teams, as well as the data science and business intelligence teams. Common use cases include analyzing web traffic patterns, troubleshooting content delivery issues, and monitoring the performance of the Akamai CDN.\n\nThis table is a fact table, contain

In [32]:
tables_info_dict_copy = tables_info_dict.copy()
columns_info_dict_copy = columns_info_dict.copy()

In [33]:
for urn in tables_info_dict.keys():
    tables_info_dict[urn]["description"] = parsed_entity_descriptions[urn][0]

In [34]:
for urn in columns_info_dict.keys():
    for column in columns_info_dict[urn].keys():
        llm_column_description = parsed_entity_descriptions[urn][1].get(column)
        actual_column_description = columns_info_dict[urn][column].get("description")
        if llm_column_description is not None:
            if actual_column_description is None or not isinstance(
                actual_column_description, str
            ):
                columns_info_dict[urn][column]["description"] = llm_column_description
            else:
                if len(llm_column_description) - len(actual_column_description) > 10:
                    columns_info_dict[urn][column][
                        "description"
                    ] = llm_column_description
                else:
                    continue
        else:
            continue

In [35]:
columns_info_dict

{'urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_lake.dl_akamai.dl_akamai_logs,PROD)': {'id': {'column_name': 'id',
   'metadata': {'nativeDataType': 'NUMBER(38,0)'},
   'upstream_lineages': [],
   'description': 'Unique identifier for the Akamai log event.'},
  'data': {'column_name': 'data',
   'metadata': {'nativeDataType': 'VARIANT'},
   'upstream_lineages': [],
   'description': 'The raw data payload from the Akamai log event, stored as a VARIANT data type.'},
  's3_filename': {'column_name': 's3_filename',
   'metadata': {'nativeDataType': 'VARCHAR(16777216)'},
   'upstream_lineages': [],
   'description': 'The name of the S3 file from which the Akamai log event was ingested.'},
  'row_hash': {'column_name': 'row_hash',
   'metadata': {'nativeDataType': 'NUMBER(38,0)'},
   'upstream_lineages': [],
   'description': 'A hash value calculated for the row, used for data quality and deduplication purposes.'},
  'dl_load_tms': {'column_name': 'dl_load_tms',
   'metadata': {'

In [36]:
with open("CARE_TABLES_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL", "wb") as f:
    pickle.dump(tables_info_dict, f)
    f.close()
with open("CARE_COLUMNS_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL", "wb") as f:
    pickle.dump(columns_info_dict, f)
    f.close()

In [40]:
for term in glossary_info.glossary.keys():
    if glossary_info.glossary[term]["term_name"] == "Gender":
        print(glossary_info.glossary[term])

{'term_name': 'Gender', 'term_description': '', 'parent_node': {'parent_name': 'PII - Level 2', 'parent_description': 'PII classified by [Care.com](//Care.com) as Level 2 which includes:  \nFull name  \nAddress information such as street address or email  \nPersonal characteristic including photos, biometrics, demographics  \nPhone Numbers  \nIP Addresses  \nFull name of family members'}, 'example': {'column_name': 'Sex', 'datatype': 'WSTR'}}


In [41]:
columns_info_dict

{'urn:li:dataset:(urn:li:dataPlatform:snowflake,prd_db_care_lake.dl_akamai.dl_akamai_logs,PROD)': {'id': {'column_name': 'id',
   'metadata': {'nativeDataType': 'NUMBER(38,0)'},
   'upstream_lineages': [],
   'description': 'Unique identifier for the Akamai log event.'},
  'data': {'column_name': 'data',
   'metadata': {'nativeDataType': 'VARIANT'},
   'upstream_lineages': [],
   'description': 'The raw data payload from the Akamai log event, stored as a VARIANT data type.'},
  's3_filename': {'column_name': 's3_filename',
   'metadata': {'nativeDataType': 'VARCHAR(16777216)'},
   'upstream_lineages': [],
   'description': 'The name of the S3 file from which the Akamai log event was ingested.'},
  'row_hash': {'column_name': 'row_hash',
   'metadata': {'nativeDataType': 'NUMBER(38,0)'},
   'upstream_lineages': [],
   'description': 'A hash value calculated for the row, used for data quality and deduplication purposes.'},
  'dl_load_tms': {'column_name': 'dl_load_tms',
   'metadata': {'